# Train balance bot using PPO with domain randomization

[![Open In Colab <](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shawnhymel/make-reinforcement-learning-for-robotics/blob/main/software/make_rl_train_balance_bot.ipynb)

Run each cell by pressing `shift + enter`.

This notebook downloads the STL models and reinforcement learning training code from [https://github.com/ShawnHymel/make-reinforcement-learning-for-robotics](https://github.com/ShawnHymel/make-reinforcement-learning-for-robotics) and trains the 2-wheeled robot to balance in three phases using domain randomization.

If you would like to deploy the agent to a real robot, you will need an [M5Stack Bala2-Fire](https://shop.m5stack.com/products/bala2-fire-self-balancing-robot-kit) balance bot kit. At the end of the training process, you will find the actor model exported to `make-reinforcement-learning-for-robotics/software/output/actor.h`. Download that file and overwrite the `actor.h` found in `make-reinforcement-learning-for-robotics/software/balance_bot_arduino`. Upload that Arduino sketch to the robot to watch it balance.

## Install and import required packages

In [ ]:
# Install required packges
%pip install \
    onnx==1.21.0 \
    onnxscript==0.7.0 \
    onnxruntime==1.25.1 \
    mujoco==3.7.0 \
    tensorboardX==2.6.5 \
    mediapy==1.2.6

In [ ]:
# Download repository for PPO algorithm and environment
!curl -L -o repo.zip https://github.com/ShawnHymel/make-reinforcement-learning-for-robotics/archive/refs/heads/main.zip
!rm -rf make-reinforcement-learning-for-robotics
!unzip -q repo.zip
!mv make-reinforcement-learning-for-robotics-main make-reinforcement-learning-for-robotics
!rm repo.zip
!ls make-reinforcement-learning-for-robotics

In [ ]:
# Move into the repo (to access the libraries)
%cd make-reinforcement-learning-for-robotics/software

In [ ]:
# Import standard libraries
import copy
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import mediapy
import numpy as np
import torch

In [ ]:
# Set MuJoCo rendering to off-screen (i.e. record video)
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

In [ ]:
# Import custom environment
from balance_bot_env import BalanceBotEnv, DomainRandomConfig

# Import PPO training module and exporter
from rl.ppo_trainer import PPOConfig, evaluate, train, export_tb_plots_as_csv, export_actor_onnx
from rl.onnx_actor_to_c import export_onnx_actor_to_c

In [ ]:
# Load TensorBoard
%load_ext tensorboard

## Settings

In [ ]:
# Training settings
MJCF_PATH = Path("../mechanical/bala2-fire/bala2-fire-simplified.xml")
SEED = 42
NUM_ENVS = 8                # Number of parallel environments
STEPS_PER_ENV = 300_000     # Number of simulation steps to perform per environment
OUTPUT_DIR = Path("output") # Where to save videos and model files

In [ ]:
# PPO settings
ppo_config = PPOConfig(
    exp_name = "balance-bot-ppo",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    actor_hidden_layers = 2,       # Number of hidden layers in the actor network
    actor_hidden_size = 32,        # Number of nodes in each hidden layer in the actor
    critic_hidden_layers = 2,      # Number of hidden layers in the critic network
    critic_hidden_size = 32,       # Number of nodes in each hidden layer in the critic
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,  # Total simulation steps (all envs and iterations)
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.000,              # Match MJCF opt.timestep for real-time rendering (or 0 for fast)
)

## Helper functions

In [ ]:
def make_balance_bot_env(**kwargs):
    """Function to create an environment for our balance bot (no rendering)"""
    # Create the environment with rendering disabled for fast training
    env = BalanceBotEnv(
        mjcf_path = MJCF_PATH,
        render_mode = None,
        **kwargs
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)

In [ ]:
def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs balance bot environments."""
    env_factories = []
    for _ in range(num_envs):
        env_factories.append(
            lambda kw=kwargs: make_balance_bot_env(**kw)
        )
    return gym.vector.SyncVectorEnv(env_factories)

In [ ]:
def record_video(
    policy=None,
    envs=None,
    video_name="video.mp4",
    num_episodes=None,
    max_steps=2000,
    fps=30,
    after_fall_s=1.0,
    seed=0,
    video_dir="videos",
    download=False,
):
    """
    Run episodes in a rendering env, show the video in the notebook, save it as an MP4, and
    report the total reward for each episode.

    Returns:
        list of float: Total reward for each episode (counted up to the fall)
    """
    # Figure out what kind of policy we have
    use_agent = policy is not None and not isinstance(policy, str)
    if num_episodes is None:
        num_episodes = 1 if use_agent else 3

    # Build a rendering env: clone a training env if given (so it has the exact same
    # settings), otherwise create a fresh env with default settings
    if envs is not None:
        env = copy.deepcopy(envs.envs[0].unwrapped)
        env.render_mode = "rgb_array"
    else:
        env = BalanceBotEnv(mjcf_path=MJCF_PATH, render_mode="rgb_array")
    env.action_space.seed(seed)

    # Grab a frame every N physics steps so the video plays in real time
    dt = env.model.opt.timestep
    frame_every = max(1, round(1 / (fps * dt)))
    extra_steps = int(after_fall_s / dt)
    if use_agent:
        device = next(policy.parameters()).device

    # Run multiple episodes
    frames, episode_returns = [], []
    for episode in range(num_episodes):
        obs, _ = env.reset(seed=seed + episode)
        frames.append(env.render())
        total_reward, fall_step = 0.0, None

        # Run a single episode
        for step in range(max_steps):
            # Choose an action from the policy
            if use_agent:
                with torch.no_grad():
                    x = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                    action = policy.actor_mean(x).cpu().numpy()[0]
                action = np.clip(action, -1.0, 1.0)

            # Choose an action randomly
            elif policy == "random":
                action = env.action_space.sample()

            # Choose no action
            else:
                action = np.zeros(env.action_space.shape, dtype=np.float32)

            # Step the sim, only counting reward up to the fall
            obs, reward, terminated, truncated, _ = env.step(action)
            if fall_step is None:
                total_reward += reward
            if step % frame_every == 0:
                frames.append(env.render())

            # Note when it tips past the limit, then keep going so we see it hit the floor
            if terminated and fall_step is None:
                fall_step = step
            if fall_step is not None and step - fall_step >= extra_steps:
                break
            if truncated:
                break

        # Report how the episode went
        if fall_step is not None:
            outcome = f"fell after {(fall_step + 1) * dt:.2f} s"
        else:
            outcome = f"still upright after {(step + 1) * dt:.2f} s"
        print(f"Episode {episode + 1}: return = {total_reward:.2f} ({outcome})")
        episode_returns.append(total_reward)

    # Close the environment
    env.close()
    if num_episodes > 1:
        print(f"Mean return: {np.mean(episode_returns):.2f}")

    # Show the video in the notebook
    mediapy.show_video(frames, fps=fps)

    # Save the video
    os.makedirs(video_dir, exist_ok=True)
    video_path = os.path.join(video_dir, video_name)
    mediapy.write_video(video_path, frames, fps=fps)
    print(f"Saved video: {video_path}")

    # Optionally download it (Colab only)
    if download:
        try:
            from google.colab import files
            files.download(video_path)
        except ImportError:
            print("Download is only available in Colab. Find the file at the path above.")

    return episode_returns

In [ ]:
def load_agent(envs, model_path):
    """
    For debugging only! Use this to load a previously trained model to skip previous phases. Note
    that you will still need to run the cells in each prior phase that update the experiment name
    and environment.
    """
    from rl.ppo_trainer import Agent, TrainResult

    # Make sure model_path is a Path
    model_path = Path(model_path)

    # Load agent from previous run
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    agent = Agent(envs, ppo_config).to(device)
    agent.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    agent.eval()
    print(f"Loaded model from {model_path}")

    # Wrap agent in a dummy result
    return TrainResult(
        agent = agent,
        checkpoint_dir = model_path.parent,
        best_model_path = model_path,
        final_model_path = None,
        best_mean_return = 0,
    )

## Test the Simulation

Let's record two videos. The first shows what happens with no policy (the robot is just allowed to fall down). The second shows a random policy, which is what the agent starts with at the beginning of training.

In [ ]:
# No agent, just let the bot fall
record_video(None, video_dir=OUTPUT_DIR, video_name="no_policy.mp4")

In [ ]:
# Random policy (choose random actions for each motor)
record_video("random", video_dir=OUTPUT_DIR, video_name="random_policy.mp4")

## Phase 1: Balance only

We'll start with the easy task of penalizing if the robot tilts (pitches) forward or turns (yaw).

In [ ]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-1"

# Create an environment that only rewards staying upright
envs = make_envs(
    NUM_ENVS,
    pitch_penalty_coef=0.5,
    action_penalty_coef=0.01,
    position_penalty_coef=0.001,
    speed_penalty_coef=20.0,
    yaw_penalty_coef=0.1,
)

In [ ]:
# Launch TensorBoard so we can watch the training process
%tensorboard --logdir runs

In [ ]:
# Choo choo train
result = train(ppo_config, envs=envs)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# Record one evaluation episode using the trained agent
returns = record_video(
    result.agent,
    envs=envs,
    video_name=ppo_config.exp_name + "_eval.mp4",
    video_dir=OUTPUT_DIR,
    download=True,
)

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(result.checkpoint_dir)

## Phase 2: Domain randomization for controls

We start with the trained agent from phase 1 and go through the same PPO training process. However, we add some variations in the observation noise, action delay, motor noise, and motor gain to simulate how these variations might show up on the real robot (such as battery voltage drop, noise, etc.).

In [ ]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-2"

# Create data randomization config to add obs noise and action delay
dr = DomainRandomConfig(
    pitch_noise_std_dev=0.01,       # Inject some noise into pitch observation
    pitch_rate_noise_std_dev=0.01,  # Inject some noise into pitch rate observation
    wheel_vel_noise_std_dev = 0.1,  # Inject some noise into wheel velocity observation
    action_delay_steps=2,           # 2 step (0-10ms) delay
    action_delay_random=True,       # Vary 0-2 steps each episode
)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [ ]:
# Launch TensorBoard so we can watch the training process
%tensorboard --logdir runs

In [ ]:
# Choo choo train (starting with the previously trained agent)
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# Record one evaluation episode using the trained agent
returns = record_video(
    result.agent,
    envs=envs,
    video_name=ppo_config.exp_name + "_eval.mp4",
    video_dir=OUTPUT_DIR,
    download=True,
)

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(result.checkpoint_dir)

## Phase 3: Domain randomization for the physical world

Repeat the training process with the agent from phase 2. This time, we randomize aspects of the phyiscal world, such as by adding random pushes to the robot, the wheel-to-ground friction, motor noise, and motor gain.

In [ ]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-3"

# Use same DR as last time, and add more variations
dr.push_prob = 0.005                    # 0.5% chance of push per step
dr.push_force_max_n = 0.3               # Gentle pushes
dr.mass_scale_range= (0.8, 1.2)         # +/-20% mass variation
dr.friction_scale_range = (0.5, 1.5)    # 50-150% friction variation
dr.motor_noise_scale = 0.02             # +/-2% motor noise
dr.motor_gain_range = (0.6, 1.0)        # 60% to 100% motor torque per episode

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [ ]:
# Launch TensorBoard so we can watch the training process
%tensorboard --logdir runs

In [ ]:
# Choo choo train (starting with the previously trained agent)
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# Record one evaluation episode using the trained agent
returns = record_video(
    result.agent,
    envs=envs,
    video_name=ppo_config.exp_name + "_eval.mp4",
    video_dir=OUTPUT_DIR,
    download=True,
)

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(result.checkpoint_dir)

## Clean up and save model

In [ ]:
# Close the environments
for idx, env in enumerate(envs.envs):
    print(f"Closing env {idx}")
    env.env.close()

In [ ]:
# Get observation and action sizes
obs_size = envs.single_observation_space.shape[0]
action_size = envs.single_action_space.shape[0]

# Export the actor network as an ONNX model
export_actor_onnx(
    model_path=result.best_model_path,
    output_path=OUTPUT_DIR / "actor.onnx",
    obs_size=obs_size,
    action_size=action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)

In [ ]:
# Export actor to .h file
export_onnx_actor_to_c(
    onnx_path   = OUTPUT_DIR / "actor.onnx",
    output_path = OUTPUT_DIR / "actor.h"
)